# A second survey: subgroup matching on the General Social Survey

The paper's main evaluation pairs one adaptation method, CultureLLM-style finetuning,
with one survey, the World Values Survey. A reviewer put it that the evidence is
narrow: one survey, four items, two countries, one main finetuning method. This
notebook is the second evaluation. It scores a second method on a second survey, on
that method's own terms: SubPOP's subgroup-matched finetuning (Suh et al.,
arXiv:2502.16761) on SubPOP-Eval, the benchmark its authors hold out, which puts 133
questions of the 2022 General Social Survey to the 22 United States subgroups SubPOP is
trained on.

The prompt is SubPOP's own: the subgroup stated as the answer to a steering question,
then the survey question with lettered options. Answers are read as next-token
probabilities over the option letters, as SubPOP reads them, so there is one
deterministic run per condition and no full-answer mode. The refusal option and the
options SubPOP marks as unordered are dropped and the rest renormalised in the order of
their levels, which is the distribution SubPOP's own Wasserstein distance scores. The
22 subgroups of a question are its cells: `group_fidelity` scores them exactly as it
scores the cells of the World Values Survey, over 231 pairs of subgroups.

In the code the conditions are the `base` and `subpop` arms of the six local backbones,
and each run sits in `outputs/culture/<model>/<arm>/gss/`, written by
`./run_experiment_add_culture.sh gss`. Everything here is written to its own files,
all named `population_fidelity_gss_*`, and drawn into `figures/fidelity/gss/`, so the
whole evaluation is removed by deleting this notebook, `culture.gss`,
`culture.fidelity_gss`, their tests, those tables and that folder.

The survey tables are gated and are not part of this repository:
`./run_experiment_add_culture.sh gss-stage` puts them under `upstream/subpop/`.


In [ ]:
from culture.fidelity_gss import (
    GSS_COMPARISON_TABLE,
    GSS_GROUPS_TABLE,
    GSS_MODELS,
    GSS_PAIRED_TABLE,
    GSS_SUMMARY_TABLE,
    build_gss,
    check_gss,
    gss_plate,
    gss_summary,
    survey_comparison,
)
from culture.fidelity_objectives import OBJECTIVES_GROUPS_TABLE, OBJECTIVES_PAIRED_TABLE
from culture.gss import GSS_ARMS, check_records, load_records, load_steering
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [ ]:
records = load_records()
check_records(records, load_steering())
groups, paired = build_gss(records)
check_gss(groups, paired)
print(len(groups), "group rows and", len(paired), "paired rows over", groups["source"].nunique(), "runs")

## The tables

`population_fidelity_gss_groups.csv` holds one row per backbone, arm and question, and
`population_fidelity_gss_paired.csv` the subgroup-matched arm beside the as-released
run of the same backbone on the subgroups both answered. `valid_answer_mass` is the
mean next-token probability the offered letters received.

`population_fidelity_gss_summary.csv` holds one row per topic, backbone and arm, with
the paper's two conventions: the level of a score across the questions is its median,
and what finetuning changed is the mean paired change, beside the counts of questions
that moved. Topic `all` is every question; `Happiness`, `Religious attendance` and
`Social trust` are the three General Social Survey questions on the attitudes the World
Values Survey evaluation also asks about.

`population_fidelity_gss_comparison.csv` sets the two evaluations side by side under
next-token probabilities, each method on its own target: the German finetuning on
German cells, the Mexican finetuning on Mexican cells and the subgroup-matched arm on
United States cells, all read from the first run of the objectives tables, then the
subgroup-matched arm on the General Social Survey. Each row pools the six backbones.

In [ ]:
summary = gss_summary(groups, paired)
wvs_groups, wvs_paired = read_csv(OBJECTIVES_GROUPS_TABLE), read_csv(OBJECTIVES_PAIRED_TABLE)
assert wvs_groups is not None and wvs_paired is not None
comparison = survey_comparison(wvs_groups, wvs_paired, groups, paired)
write_csv(groups, GSS_GROUPS_TABLE)
write_csv(paired, GSS_PAIRED_TABLE)
write_csv(summary, GSS_SUMMARY_TABLE)
write_csv(comparison, GSS_COMPARISON_TABLE)
print(len(summary), "summary rows;", len(comparison), "comparison rows")

## The comparison

Mean paired change from the as-released model, with the count of comparisons in which
PFS rises, falls or stays at zero, the count in which center alignment rises, and the
count in which structure is the smallest term.

In [ ]:
changes = [f"delta_{score}_vs_base_mean" for score in ("score_accuracy", "score_dispersion", "score_structure", "pfs", "score_center")]
counts = ["n_paired", "n_pfs_up", "n_pfs_down", "n_pfs_same", "n_center_up", "n_structure_binding"]
for topic, block in comparison.groupby("topic", sort=False):
    print(topic)
    print(block.set_index(["survey", "cells", "arm"])[[*changes, *counts]].round(3).to_string(), "\n")

## Every backbone

The median of each score over the questions, one row per backbone and arm, with the
mean answer mass on the offered letters. A mass under 0.10 would mark a run whose
letters say little about the model.

In [ ]:
pooled = summary[summary["topic"].eq("all")].set_index(["model_key", "arm"])
medians = [f"{score}_median" for score in ("score_accuracy", "adaptability_ratio", "score_structure", "pfs", "score_center")]
order = [(key, arm) for key in GSS_MODELS for arm in GSS_ARMS if (key, arm) in pooled.index]
print(pooled.loc[order, [*medians, "valid_answer_mass_mean", "delta_pfs_vs_base_mean", "n_pfs_up", "n_pfs_down"]].round(3).to_string())

## The plate

One figure in the layout of the six-model component plate: five panels, one per score,
and one marker per backbone in each row, hollow for the as-released model and half
filled for the subgroup-matched one. The first block is the median over every question,
the others the three questions on the attitudes both surveys ask about.

In [ ]:
files = gss_plate(groups)
print(len(files), "files")